
## LeetCode Style Exercise: Min Cost to Connect Cities

### Problem Description
You are given an array `points` representing integer coordinates of $n$ cities on a 2D-plane, where $points[i] = [x_i, y_i]$.

The cost of connecting two cities $[x_1, y_1]$ and $[x_2, y_2]$ is the Manhattan distance: $|x_1 - x_2| + |y_1 - y_2|$.

Return the minimum cost to make all cities connected. This is a Minimum Spanning Tree (MST) problem.

### Example
Input: `points = [[0,0],[2,2],[3,10],[5,2],[7,0]]`
Output: `20`

---

## Two Valid Paths (You Can Choose)

1. **Kruskal + DSU**: build/sort edges globally.
2. **Prim + Min-Heap**: keep a dynamic ranking of the next best edge.

In interviews, presenting both options is a strong signal.

### 1) Specification (Engineering View)

**Goal:** Connect $N$ nodes with exactly $N-1$ edges minimizing total Manhattan cost.

**Blueprint (type hints):**

```python
from typing import List

def min_cost_connect_points(points: List[List[int]], method: str = "kruskal") -> int:
    ...
```

**Method options:**
1. `method="kruskal"`: DSU + sorted edges.
2. `method="prim_heap"`: min-heap ranking from the start.

**Edge Cases:**
1. $N = 1$ -> answer is 0.
2. Duplicate points -> zero-cost edges are valid.
3. Dense graph -> complete graph has $E = \frac{N(N-1)}{2}$.

---

### 2) Plan (Kruskal vs Heap Ranking)

**Kruskal + DSU:**
1. Generate all edges.
2. Sort by cost.
3. Add edge if `union(u, v)` succeeds.

**Prim + Min-Heap (ranked):**
1. Start from one city with cost 0 in heap.
2. Pop smallest candidate edge (top ranked).
3. Add this city to MST, then push/update candidates to unvisited cities.
4. Repeat until all cities are connected.

Heap intuition: the min-heap is a live ranking of the next best connection.

---

### 3) Complexity

- Kruskal: time $O(N^2 \log N)$, space $O(N^2)$ (stores all edges).
- Prim + heap (this implementation): time $O(N^2 \log N)$, space up to $O(N^2)$ due to heap candidates.

If memory is tight, Prim without storing all edges at once is usually more practical than Kruskal.

In [ ]:
from typing import List
import heapq

class DSU:
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, node: int) -> int:
        if self.parent[node] == node:
            return node
        self.parent[node] = self.find(self.parent[node])
        return self.parent[node]

    def union(self, a: int, b: int) -> bool:
        root_a = self.find(a)
        root_b = self.find(b)

        if root_a == root_b:
            return False

        if self.rank[root_a] < self.rank[root_b]:
            self.parent[root_a] = root_b
        elif self.rank[root_a] > self.rank[root_b]:
            self.parent[root_b] = root_a
        else:
            self.parent[root_b] = root_a
            self.rank[root_a] += 1

        return True

def min_cost_connect_points_kruskal(points: List[List[int]]) -> int:
    n = len(points)
    if n <= 1:
        return 0

    edges = []
    for i in range(n):
        for j in range(i + 1, n):
            dist = abs(points[i][0] - points[j][0]) + abs(points[i][1] - points[j][1])
            edges.append((dist, i, j))

    edges.sort()
    dsu = DSU(n)
    total_cost = 0
    edges_used = 0

    for cost, u, v in edges:
        if dsu.union(u, v):
            total_cost += cost
            edges_used += 1
            if edges_used == n - 1:
                break

    return total_cost

def min_cost_connect_points_prim_heap(points: List[List[int]]) -> int:
    n = len(points)
    if n <= 1:
        return 0

    in_mst = [False] * n
    min_heap = [(0, 0)]  # (cost, city)
    total_cost = 0
    used = 0

    while used < n:
        cost, city = heapq.heappop(min_heap)
        if in_mst[city]:
            continue

        in_mst[city] = True
        total_cost += cost
        used += 1

        x1, y1 = points[city]
        for nxt in range(n):
            if not in_mst[nxt]:
                x2, y2 = points[nxt]
                dist = abs(x1 - x2) + abs(y1 - y2)
                heapq.heappush(min_heap, (dist, nxt))

    return total_cost

def min_cost_connect_points(points: List[List[int]], method: str = "kruskal") -> int:
    if method == "kruskal":
        return min_cost_connect_points_kruskal(points)
    if method == "prim_heap":
        return min_cost_connect_points_prim_heap(points)
    raise ValueError("method must be 'kruskal' or 'prim_heap'")

# Example
sample = [[0, 0], [2, 2], [3, 10], [5, 2], [7, 0]]
print(min_cost_connect_points(sample, method="kruskal"))
print(min_cost_connect_points(sample, method="prim_heap"))